# Exploratory Appendix

Variables from the original notebook that are outside the pre-registered question: wind, LAI, and seasonal climatology.
Nothing here feeds the confirmatory analysis.
Requires the optional extras: `uv sync --extra appendix`.

In [ ]:
import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from windrose import WindroseAxes

from wapusk_greenup import aoi, config, extract, gee

gee.initialize()
cfg = config.load()

## Daily Park-Mean Wind from ERA5-Land

Pulled year by year to stay under `getInfo` limits.

In [ ]:
def era5_wind(year: int) -> pd.DataFrame:
    coll = ee.ImageCollection(cfg["datasets"]["reanalysis"]).filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    bands = ["u_component_of_wind_10m", "v_component_of_wind_10m"]

    def reduce(img):
        stats = img.select(bands).reduceRegion(ee.Reducer.mean(), aoi.boundary(), scale=11132)
        return ee.Feature(None, stats).set("date", img.date().format("YYYY-MM-dd"))

    df = pd.DataFrame([f["properties"] for f in coll.map(reduce).getInfo()["features"]])
    df["date"] = pd.to_datetime(df["date"])
    return df.rename(columns={bands[0]: "u", bands[1]: "v"})


wind = pd.concat([era5_wind(y) for y in cfg["year_list"]], ignore_index=True)
wind["speed"] = (wind["u"] ** 2 + wind["v"] ** 2) ** 0.5
# Meteorological convention: the direction the wind blows from, degrees clockwise from north.
wind["direction"] = (180 + np.degrees(np.arctan2(wind["u"], wind["v"]))) % 360 if False else (180 + __import__("numpy").degrees(__import__("numpy").arctan2(wind["u"], wind["v"]))) % 360

## Wind Roses by Season

In [ ]:
seasons = {"Winter": [12, 1, 2], "Spring": [3, 4, 5], "Summer": [6, 7, 8], "Fall": [9, 10, 11]}
for name, months in seasons.items():
    sub = wind[wind["date"].dt.month.isin(months)]
    ax = WindroseAxes.from_ax()
    ax.bar(sub["direction"], sub["speed"], normed=True, opening=0.8, edgecolor="white")
    ax.set_legend(title="Wind speed (m/s)")
    ax.set_title(f"{name} wind, ERA5-Land park mean, 2001-2024")
    plt.show()

## Seasonal Climatology from Cached ERA5-Land

In [ ]:
era = extract.load("era5")
era["doy"] = era["date"].dt.dayofyear
clim = era.groupby("doy")[["t2m_c", "snow_cover", "snow_depth_cm"]].mean()
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
for ax, col, label in zip(axes, clim.columns, ["2 m temperature (C)", "Snow cover (%)", "Snow depth (cm)"]):
    ax.plot(clim.index, clim[col])
    ax.set_ylabel(label)
axes[-1].set_xlabel("Day of year")
plt.show()